# N5 — Capstone: Putting It All Together

### An interactive bird's-eye-view (BEV) autonomy simulator

Across N1–N4 we built the pieces of a self-driving stack in isolation. This
capstone fuses them into **one closed-loop simulation** you can poke at: an ego
vehicle drives a **real KITTI trajectory** through a BEV scene populated with
other cars, perceives them, predicts collisions, and either brakes in time — or
crashes. Every knob on the dashboard at the bottom ties back to a notebook:

| Earlier notebook | Its role in this capstone |
|---|---|
| **N1** — Camera–LiDAR Projection | Defines the **metric BEV ground-plane frame** everything lives in; the optional bonus cell back-projects real detections into it |
| **N2** — Kalman From Scratch | The **constant-velocity Kalman filter** behind every tracked car; **coasting** when a sensor drops out |
| **N3** — LiDAR 3D Tracking | **Per-object tracking** — Hungarian association with IDs and a tentative→confirmed→coasting lifecycle, lifted into the BEV plane |
| **N4** — Path Tracking & Control | The **bicycle model + path-tracking controller** (pure-pursuit / Stanley) that actually steers the ego |
| **+ new** | A **time-to-collision (TTC) safety layer** — the "crash vs. avoid" switch |

**The payoff:** at the end you get a dashboard where you tune the controller,
toggle collision avoidance, script a troublemaking car, and inject sensor noise
— and watch, in BEV, whether the ego survives.

## 1. Setup

In [ ]:
%pip install numpy scipy matplotlib ipywidgets pykitti --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from matplotlib.transforms import Affine2D
from IPython.display import HTML, display
from scipy.optimize import linear_sum_assignment
from dataclasses import dataclass
import os, glob

np.random.seed(0)
print("Libraries loaded.")

## 2. The shared world: a BEV frame anchored on real KITTI data

Everything in this notebook lives in a single **bird's-eye-view metric frame** —
X points forward, Y points left, units are meters. This is the same ground plane
N1 back-projects into and the same East-North convention N3 used for KITTI's
GPS/IMU (OxTS) data.

The **ego vehicle's reference path is the real trajectory** the KITTI car drove
on drive `2011_09_26_0005` — parsed from OxTS latitude/longitude exactly as in
N3. If KITTI isn't available we fall back to a representative synthetic course
(straight → sweeping curve → straight) so the notebook always runs.

---

### From GPS coordinates to a local metric frame

KITTI's OxTS unit logs **latitude $\phi$ and longitude $\lambda$**, not meters. We convert each fix to a local **East–North** frame centered on the first sample $(\phi_0, \lambda_0)$ with the equirectangular (flat-earth) approximation:

$$x_E = (\lambda - \lambda_0)\,\cos\phi_0 \cdot R_{\deg}, \qquad y_N = (\phi - \phi_0)\cdot R_{\deg}$$

where $R_{\deg} \approx 111{,}320$ m is the length of one degree of latitude, and the $\cos\phi_0$ factor accounts for longitude lines bunching together away from the equator. This is accurate to centimeters over the few-kilometer span of one drive — the same conversion N3 used for its KITTI example.

We then **re-sample the trajectory to a uniform arclength spacing** $\Delta s \approx 0.5$ m (`densify_path`), so lookahead and Frenet lookups behave consistently no matter how fast the car was driving when each sample was logged.

In [ ]:
def normalize_angle(a):
    """Wrap angle to [-pi, pi]."""
    return (a + np.pi) % (2 * np.pi) - np.pi


def load_kitti_ego_path(base="kitti_data", date="2011_09_26", drive="0005", ds=0.5):
    """Real ego path from KITTI OxTS (lat/lon -> local East-North meters).
    Returns a densified (N,2) array of [x_forward, y_left] waypoints, or None."""
    oxts_dir = os.path.join(base, date, f"{date}_drive_{drive}_sync", "oxts", "data")
    files = sorted(glob.glob(os.path.join(oxts_dir, "*.txt")))
    if not files:
        return None
    oxts = np.array([np.loadtxt(f) for f in files])
    lat0 = np.radians(oxts[0, 0])
    east = (oxts[:, 1] - oxts[0, 1]) * np.cos(lat0) * 111320.0
    north = (oxts[:, 0] - oxts[0, 0]) * 111320.0
    raw = np.column_stack([east, north])          # x=east(forward-ish), y=north(left-ish)
    return densify_path(raw, ds)


def synthetic_path(ds=0.5):
    """Fallback: straight (60 m) -> gentle left curve -> straight (60 m)."""
    segs = []
    n1 = int(60 / ds)
    segs.append(np.column_stack([np.linspace(0, 60, n1), np.zeros(n1)]))
    R, arc = 80.0, 40.0
    th = np.linspace(0, arc / R, int(arc / ds))
    cx, cy = 60.0, R
    sx, sy = cx + R * np.sin(th), cy - R * np.cos(th)
    segs.append(np.column_stack([sx, sy]))
    ex, ey, hd = sx[-1], sy[-1], arc / R
    n3 = int(60 / ds)
    segs.append(np.column_stack([ex + np.linspace(0, 60 * np.cos(hd), n3),
                                 ey + np.linspace(0, 60 * np.sin(hd), n3)]))
    return np.vstack(segs)


def densify_path(raw, ds=0.5):
    """Resample a coarse polyline to ~uniform ds spacing."""
    seg = np.hypot(np.diff(raw[:, 0]), np.diff(raw[:, 1]))
    s = np.concatenate([[0], np.cumsum(seg)])
    n = max(int(s[-1] / ds), 2)
    su = np.linspace(0, s[-1], n)
    return np.column_stack([np.interp(su, s, raw[:, 0]), np.interp(su, s, raw[:, 1])])


# Try real KITTI; fall back to synthetic
PATH = load_kitti_ego_path()
if PATH is None:
    print("KITTI OxTS not found - using synthetic reference path.")
    PATH = synthetic_path()
else:
    print(f"Loaded real KITTI ego path: {len(PATH)} waypoints.")

print(f"Path extent: X [{PATH[:,0].min():.0f}, {PATH[:,0].max():.0f}] m, "
      f"Y [{PATH[:,1].min():.0f}, {PATH[:,1].max():.0f}] m")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(PATH[:, 0], PATH[:, 1], "-", color="gray", lw=2)
ax.plot(PATH[0, 0], PATH[0, 1], "go", ms=10, label="start")
ax.set_aspect("equal"); ax.grid(alpha=0.3)
ax.set_xlabel("X — forward (m)"); ax.set_ylabel("Y — left (m)")
ax.set_title("Ego reference path (BEV)"); ax.legend()
plt.show()

## 3. Lanes that follow the road: the Frenet frame

Other cars don't drive in straight world-lines — they follow the road. We place
each agent in **path-relative (Frenet) coordinates**: an arclength `s` along the
reference path and a lateral offset `d` (positive = left of the path). A lane is
then just a constant `d`, and it stays a lane even when the road curves. A
"cut-in" becomes a smooth change in `d`; the ego's lane is `d = 0`.

---

### The Frenet transform

Let the reference path be a curve $\mathbf{r}(s)$ parameterized by arclength $s$. At each point it has a unit **tangent** and a unit **left normal**:

$$\hat{\mathbf{t}}(s) = \begin{bmatrix}\cos\psi(s)\\ \sin\psi(s)\end{bmatrix}, \qquad \hat{\mathbf{n}}(s) = \begin{bmatrix}-\sin\psi(s)\\ \cos\psi(s)\end{bmatrix}$$

where $\psi(s)$ is the local path heading. A Frenet coordinate $(s, d)$ maps to the world by walking $s$ along the path and stepping $d$ along the left normal:

$$\mathbf{p}(s, d) = \mathbf{r}(s) + d\,\hat{\mathbf{n}}(s)$$

This is exactly what `frenet_to_world` computes: find the segment containing $s$, take its heading, and offset by $d$ along $[-\sin\psi,\ \cos\psi]$. A car holding a constant $d$ therefore traces a path **parallel to the road**, curving along with it.

**Validity:** the map is one-to-one only while $|d| < 1/\kappa(s)$ — the lateral offset stays smaller than the local radius of curvature $1/\kappa$. For real lanes ($d \sim 3.5$ m) on road-scale curves ($1/\kappa \sim 80$ m) we are nowhere near violating it.

In [ ]:
def path_arclength(path):
    seg = np.hypot(np.diff(path[:, 0]), np.diff(path[:, 1]))
    return np.concatenate([[0.0], np.cumsum(seg)])


def path_frame(path, idx):
    """(origin xy, tangent heading) at a path index."""
    i = int(np.clip(idx, 0, len(path) - 2))
    d = path[i + 1] - path[i]
    return path[i], np.arctan2(d[1], d[0])


def frenet_to_world(path, s_arr, s, d):
    """(arclength s, lateral offset d) -> world [x, y, heading]."""
    i = int(np.clip(np.searchsorted(s_arr, s) - 1, 0, len(path) - 2))
    seg = path[i + 1] - path[i]
    h = np.arctan2(seg[1], seg[0])
    along = (s - s_arr[i]) / max(s_arr[i + 1] - s_arr[i], 1e-9)
    base = path[i] + along * seg
    nrm = np.array([-np.sin(h), np.cos(h)])   # left normal
    p = base + d * nrm
    return p[0], p[1], h

print("Frenet helpers ready.")

## 4. The ego vehicle — bicycle model + path tracking (N4)

The ego is driven by N4's **kinematic bicycle model** and a **pure-pursuit** (or
**Stanley**) controller that steers toward the reference path. Nothing new here —
we are reusing N4 directly. The controller is the first knob participants will
tune (lookahead distance, controller type, target speed).

---

### Recap of the equations (from N4)

**Kinematic bicycle model** — state $[x, y, \psi, v]$, controls steering $\delta$ and acceleration $a$, wheelbase $L$:

$$\dot{x} = v\cos\psi, \quad \dot{y} = v\sin\psi, \quad \dot{\psi} = \frac{v}{L}\tan\delta, \quad \dot{v} = a$$

stepped forward by $\Delta t$ in `bicycle_step`.

**Pure pursuit** — aim at a lookahead point $(x_L, y_L)$ a distance $L_d$ ahead; with $\alpha$ the angle from the heading to that point, the steering that fits a circular arc through the rear axle and the point is:

$$\alpha = \operatorname{atan2}(y_L - y,\; x_L - x) - \psi, \qquad \delta = \arctan\!\left(\frac{2L\sin\alpha}{L_d}\right)$$

**Stanley** — cancel heading error $\psi_{\text{path}}-\psi$ and cross-track error $e$ (gain $k$, speed $v$):

$$\delta = (\psi_{\text{path}} - \psi) + \arctan\!\left(\frac{k\,e}{v}\right)$$

The lookahead $L_d$ (pure pursuit) and gain $k$ (Stanley) are the two steering knobs on the dashboard. See N4 for the full derivation and tuning sweep.

In [ ]:
L_WB = 2.5                      # wheelbase (m)
MAX_STEER = np.radians(35)

def bicycle_step(state, delta, a, dt, L=L_WB):
    """One step of the kinematic bicycle model. state = [x, y, psi, v]."""
    x, y, psi, v = state
    x += v * np.cos(psi) * dt
    y += v * np.sin(psi) * dt
    psi = normalize_angle(psi + v / L * np.tan(delta) * dt)
    v = max(v + a * dt, 0.0)
    return np.array([x, y, psi, v])


def find_lookahead_point(state, path, Ld):
    x, y = state[0], state[1]
    d = np.hypot(path[:, 0] - x, path[:, 1] - y)
    nearest = int(np.argmin(d))
    for i in range(nearest, len(path)):
        if np.hypot(path[i, 0] - x, path[i, 1] - y) >= Ld:
            return path[i], nearest
    return path[-1], nearest


def pure_pursuit_steering(state, la_pt, L=L_WB):
    x, y, psi, v = state
    alpha = normalize_angle(np.arctan2(la_pt[1] - y, la_pt[0] - x) - psi)
    Ld = max(np.hypot(la_pt[0] - x, la_pt[1] - y), 0.1)
    return np.clip(np.arctan2(2 * L * np.sin(alpha), Ld), -MAX_STEER, MAX_STEER)


def stanley_steering(state, path, k, L=L_WB):
    x, y, psi, v = state
    fx, fy = x + L * np.cos(psi), y + L * np.sin(psi)
    d = np.hypot(path[:, 0] - fx, path[:, 1] - fy)
    i = int(np.argmin(d)); j = min(i + 1, len(path) - 1)
    pdx, pdy = path[j] - path[max(i - 1, 0)]
    he = normalize_angle(np.arctan2(pdy, pdx) - psi)
    cte = (pdx * (fy - path[i, 1]) - pdy * (fx - path[i, 0])) / (np.hypot(pdx, pdy) + 1e-9)
    return np.clip(he + np.arctan2(k * (-cte), max(abs(v), 0.5)), -MAX_STEER, MAX_STEER)


def cross_track_error(state, path):
    x, y = state[0], state[1]
    d = np.hypot(path[:, 0] - x, path[:, 1] - y)
    i = int(np.argmin(d)); j = min(i + 1, len(path) - 1)
    pdx, pdy = path[j] - path[max(i - 1, 0)]
    return (pdx * (y - path[i, 1]) - pdy * (x - path[i, 0])) / (np.hypot(pdx, pdy) + 1e-9)

print("Ego model + controllers ready (from N4).")

## 5. The other cars — constant-velocity motion

Each other vehicle advances along its lane at constant speed (the simple case of
the CTRV turning model). The **antagonist** is the troublemaker: in the `cut_in`
scenario it starts one lane to the left and smoothly merges into the ego's lane;
in `sudden_stop` it sits ahead in the ego's lane and slams on the brakes. The
**ambient** car is harmless scenery in the left lane.

---

### Lane-relative motion: the zero-yaw-rate case of CTRV

Each agent advances at **constant speed along its lane**, which in Frenet coordinates is simply:

$$s_{k+1} = s_k + v\,\Delta t, \qquad d_{k+1} = d_k \;\;(\text{constant, except during a cut-in})$$

This is the **$\dot\psi = 0$ special case of the CTRV turning model** — yet the car's *world* trajectory is still curved, because the lane itself follows the road. The curvature comes from the path, so we don't need an explicit yaw-rate state here. (The full CTRV model — with an explicit yaw-rate state — is what you would reach for to track a free GPS/IMU trajectory that has no lane to ride on.)

**Sudden stop** decelerates at the braking limit: $\;v_{k+1} = \max(v_k - a_{\text{brake}}\,\Delta t,\; 0)$.

**Cut-in** slides the lateral offset from $d_0$ to $d_1$ with a **smoothstep** (raised cosine) over $1.8$ s:

$$d(t) = d_0 + (d_1 - d_0)\cdot\tfrac{1}{2}\big(1 - \cos(\pi u)\big), \qquad u = \operatorname{clip}\!\left(\frac{t - t_{\text{trig}}}{1.8},\, 0,\, 1\right)$$

The smoothstep is $C^1$-continuous — lateral velocity ramps up and back to zero — so the merge looks like a real lane change rather than an instantaneous teleport.

In [ ]:
def lane_change_offset(scenario, t, trigger, d0, d1):
    """Smooth (smoothstep) lateral lane change from d0 to d1 over 1.8 s."""
    if scenario != "cut_in" or t < trigger:
        return d0
    u = np.clip((t - trigger) / 1.8, 0.0, 1.0)
    return d0 + (d1 - d0) * (0.5 - 0.5 * np.cos(np.pi * u))

print("Agent motion ready (CTRV / lane-relative).")

## 6. Perception in the loop — a BEV tracker (N2 + N3)

The ego doesn't get the agents' true positions for free. It **observes** them
through a noisy sensor and must *track* them. We reuse N3's tracker design — a
**per-object Kalman filter** (N2's constant-velocity model `[x, y, vx, vy]`) plus
**Hungarian association** with distance gating and a **tentative→confirmed→
coasting→dead** lifecycle — but in the BEV plane instead of the image plane.

When the sensor drops out, matched detections disappear and confirmed tracks
**coast** on their Kalman prediction (exactly the behavior from N2), growing more
uncertain until measurements return.

---

### The per-object filter (N2), in the BEV plane

Each track runs a **constant-velocity Kalman filter** with state $\mathbf{x} = [x, y, v_x, v_y]^\top$, predicted with a constant-velocity model and corrected by position-only measurements $\mathbf{z} = [x, y]^\top$:

$$F = \begin{bmatrix} 1 & 0 & \Delta t & 0 \\ 0 & 1 & 0 & \Delta t \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}, \qquad H = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \end{bmatrix}$$

**Predict:** $\quad \mathbf{x}^- = F\mathbf{x}, \qquad P^- = FPF^\top + Q$

**Update:** $\quad \mathbf{y} = \mathbf{z} - H\mathbf{x}^-, \quad S = HP^-H^\top + R, \quad K = P^-H^\top S^{-1}$

$$\mathbf{x}^+ = \mathbf{x}^- + K\mathbf{y}, \qquad P^+ = (I - KH)\,P^-$$

with process noise $Q = \operatorname{diag}(0.5, 0.5, 2, 2)$ and measurement noise $R = \operatorname{diag}(1, 1)$. **Coasting** is just running `predict()` with no `update()`: $P$ grows every step, so the uncertainty ellipse visibly inflates while the sensor is out.

### Association in metric BEV

Just as in N3, we track **points in meters** (not image boxes), so the cost is plain Euclidean distance between each track's predicted position $\hat{\mathbf{p}}_i$ and each detection $\mathbf{z}_j$:

$$C_{ij} = \big\lVert \hat{\mathbf{p}}_i - \mathbf{z}_j \big\rVert_2$$

The Hungarian algorithm picks the minimum-cost assignment, and a **gate** of $4$ m rejects any matched pair farther apart than that (a physically implausible jump), returning both to the unmatched pools — the same distance gating as N3, in metric units.

In [ ]:
class BevTrack:
    """One tracked vehicle: constant-velocity Kalman filter (N2) + lifecycle (N3)."""
    _next = 1
    def __init__(self, z, dt):
        self.id = BevTrack._next; BevTrack._next += 1
        rng = np.random.RandomState(self.id * 7 + 13)
        self.color = tuple(rng.uniform(0.2, 0.9, 3))
        self.F = np.array([[1,0,dt,0],[0,1,0,dt],[0,0,1,0],[0,0,0,1]], float)
        self.H = np.array([[1,0,0,0],[0,1,0,0]], float)
        self.Q = np.diag([0.5, 0.5, 2.0, 2.0])
        self.R = np.diag([1.0, 1.0])
        self.x = np.array([z[0], z[1], 0.0, 0.0])
        self.P = np.diag([2.0, 2.0, 10.0, 10.0])
        self.hits = 1; self.misses = 0; self.state = "tentative"
    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
    def update(self, z):
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P
        self.hits += 1; self.misses = 0
    def pos(self): return self.x[:2]
    def vel(self): return self.x[2:]


class BevTracker:
    """SORT-style tracker in BEV: predict -> Hungarian assign -> update -> birth/death."""
    def __init__(self, dt, gate=4.0, min_hits=2, max_misses=5):
        self.dt = dt; self.gate = gate
        self.min_hits = min_hits; self.max_misses = max_misses
        self.tracks = []
    def step(self, detections):
        for t in self.tracks:
            t.predict()
        dets = np.array(detections) if len(detections) else np.empty((0, 2))
        matched_t, matched_d = set(), set()
        if self.tracks and len(dets):
            C = np.zeros((len(self.tracks), len(dets)))
            for i, t in enumerate(self.tracks):
                C[i] = np.hypot(dets[:, 0] - t.pos()[0], dets[:, 1] - t.pos()[1])
            for i, j in zip(*linear_sum_assignment(C)):
                if C[i, j] <= self.gate:
                    self.tracks[i].update(dets[j]); matched_t.add(i); matched_d.add(j)
        for i, t in enumerate(self.tracks):
            if i not in matched_t:
                t.misses += 1
                if t.state == "confirmed":
                    t.state = "coasting"
            elif t.hits >= self.min_hits:
                t.state = "confirmed"
        for j in range(len(dets)):
            if j not in matched_d:
                self.tracks.append(BevTrack(dets[j], self.dt))
        self.tracks = [t for t in self.tracks if t.misses <= self.max_misses]
        return [t for t in self.tracks if t.state in ("confirmed", "coasting")]

print("BEV tracker ready (N2 Kalman + N3 lifecycle).")

## 7. The new piece — collision forecasting and a safety layer

This is the glue that turns perception + control into *decisions*. Each step we
**roll the future forward**: the ego at its current heading and speed, and each
tracked car along its estimated velocity (a short CTRV/constant-velocity rollout).
If their bounding circles will overlap, we record the **time-to-collision (TTC)**.

The safety layer combines two behaviors:
- **Adaptive cruise** — keep a safe time-gap behind the car ahead in our lane.
- **Emergency brake** — if TTC drops below the threshold, brake hard.

Toggle the safety layer off and the ego ignores all of this — the perfect way to
*see* why it matters.

---

### Time-to-collision (TTC)

We roll the ego and every tracked car forward at **constant velocity** and look for the first instant their safety circles touch. With the ego frozen at heading $\psi$ and speed $v$, and track $i$ at $\mathbf{o}_i$ moving with velocity $\mathbf{v}_i$:

$$\mathbf{e}(\tau) = \mathbf{e}_0 + \tau v\begin{bmatrix}\cos\psi\\ \sin\psi\end{bmatrix}, \qquad \mathbf{o}_i(\tau) = \mathbf{o}_i + \tau\,\mathbf{v}_i$$

$$\mathrm{TTC} = \min\Big\{\tau \in (0,\, H] \;:\; \exists\, i,\ \lVert \mathbf{e}(\tau) - \mathbf{o}_i(\tau)\rVert < r_{\text{crash}}\Big\}$$

and $\mathrm{TTC} = \infty$ if no overlap occurs within the horizon $H$. `forecast_ttc` evaluates this on a $\Delta t$ time grid out to $H = 4$ s. (Note this ignores ego steering — a deliberately conservative, cheap forecast.)

### Adaptive cruise + emergency brake

The **lead car** is the nearest track ahead within the ego's lane, found by projecting its relative position onto the ego heading ($\text{fwd} > 0$) and the left axis ($|\text{lat}| < 2$ m). The cruise law holds a speed-dependent gap:

$$g^* = g_0 + \tau_h\,v \quad (g_0 = 6\ \text{m},\ \tau_h = 1.2\ \text{s}), \qquad a = k_g\,(g - g^*) + k_v\,(v_{\text{lead}} - v)$$

with $k_g = 0.6,\ k_v = 1.0$: the first term closes the gap error, the second matches the lead's speed. Above it sits a hard override — **if $\mathrm{TTC} <$ threshold, command full braking** $a = -a_{\max}$, ignoring the cruise term. (These gains are illustrative, chosen to behave well here — not tuned for a formal stability guarantee.)

This forward extrapolation is the **same one you saw as prediction arrows in N3** — here it is collapsed into a single number, the time-to-collision (TTC).

In [ ]:
def forecast_ttc(ego_state, tracks, horizon, dt, crash_dist):
    """Min time-to-collision over all tracks (inf if no collision predicted)."""
    ex, ey, epsi, ev = ego_state[0], ego_state[1], ego_state[2], ego_state[3]
    steps = int(horizon / dt)
    best = np.inf
    for t in tracks:
        px, py = t.pos(); vx, vy = t.vel()
        for k in range(1, steps + 1):
            tt = k * dt
            egx, egy = ex + ev*np.cos(epsi)*tt, ey + ev*np.sin(epsi)*tt
            if np.hypot(egx - (px + vx*tt), egy - (py + vy*tt)) < crash_dist:
                best = min(best, tt); break
    return best

print("Collision forecasting ready.")

## 8. The closed loop — `simulate()`

Now we wire it all together into one **pure, deterministic** function: given a
`SimConfig` (all the knobs) it runs the closed loop and returns a log of
everything that happened — no plotting. Each step:

1. **Agents move** (lane-relative, constant velocity).
2. **Sense** them with noise (skipped during a dropout window).
3. **Track** them (BEV Kalman tracker).
4. **Forecast** TTC and choose acceleration (cruise / brake) + steering (N4).
5. **Step** the ego's bicycle model and **check for a crash**.

---

### One discrete-time step

The loop is plain **forward-Euler integration** at a fixed $\Delta t = 0.1$ s (10 Hz): each tick perceives, decides, then advances the true ego state with `bicycle_step`. A crash is logged the first time the true ego comes within $r_{\text{crash}}$ of any agent. The function is **seeded and deterministic** — identical `SimConfig` in, identical log out — which is what lets the dashboard cache a run and scrub through time without re-simulating.

In [ ]:
@dataclass
class SimConfig:
    controller: str = "pure_pursuit"   # pure_pursuit | stanley
    lookahead: float = 8.0
    target_speed: float = 8.0
    stanley_k: float = 2.0
    avoidance: bool = True
    ttc_threshold: float = 3.0
    scenario: str = "cut_in"           # cut_in | sudden_stop | none
    antagonist_trigger: float = 5.0
    antagonist_speed: float = 6.0
    sensor_noise: float = 0.4
    dropout_start: float = -1.0        # <0 disables
    dropout_end: float = -1.0
    dt: float = 0.1
    duration: float = 22.0
    seed: int = 0
    horizon: float = 4.0
    crash_dist: float = 3.0
    max_decel: float = 5.0
    max_accel: float = 2.0


def simulate(cfg, path=PATH):
    rng = np.random.RandomState(cfg.seed)
    BevTrack._next = 1
    s_arr = path_arclength(path)
    n = int(cfg.duration / cfg.dt)

    p0, h0 = path_frame(path, 0)
    ego = np.array([p0[0], p0[1], h0, cfg.target_speed])
    ego_est = ego.copy()                       # ego's believed state (degrades under dropout)

    antag_d0 = 0.0 if cfg.scenario == "sudden_stop" else 3.5
    antag = {"s": 15.0, "d": antag_d0, "v": cfg.antagonist_speed}
    ambient = {"s": 40.0, "d": 3.5, "v": cfg.target_speed + 3.0}

    def agent_world(a):
        x, y, h = frenet_to_world(path, s_arr, a["s"], a["d"])
        return np.array([x, y, h, a["v"]])

    tracker = BevTracker(cfg.dt)
    log = {k: [] for k in ("t","ego","ego_est","antag","ambient","tracks",
                            "ttc","min_dist","cte","braking","crash")}
    crashed = False

    for k in range(n):
        t = k * cfg.dt

        # 1. agents move
        if cfg.scenario == "sudden_stop" and t >= cfg.antagonist_trigger:
            antag["v"] = max(antag["v"] - cfg.max_decel * cfg.dt, 0.0)
        antag["d"] = lane_change_offset(cfg.scenario, t, cfg.antagonist_trigger, antag_d0, 0.0)
        antag["s"] += antag["v"] * cfg.dt
        ambient["s"] += ambient["v"] * cfg.dt
        antag_w, ambient_w = agent_world(antag), agent_world(ambient)
        agents = [antag_w, ambient_w]

        # 2. sense (with dropout)
        in_dropout = cfg.dropout_start >= 0 and cfg.dropout_start <= t < cfg.dropout_end
        dets = []
        if not in_dropout:
            for a in agents:
                dets.append([a[0] + rng.normal(0, cfg.sensor_noise),
                             a[1] + rng.normal(0, cfg.sensor_noise)])
        # 3. track
        tracks = tracker.step(dets)

        # ego self-estimate: smooth GPS, or coast during dropout (N2 behavior)
        if not in_dropout:
            gps = ego[:2] + rng.normal(0, cfg.sensor_noise, 2)
            ego_est[:2] = 0.6 * ego_est[:2] + 0.4 * gps
        else:
            ego_est[0] += ego_est[3] * np.cos(ego_est[2]) * cfg.dt
            ego_est[1] += ego_est[3] * np.sin(ego_est[2]) * cfg.dt
        ego_est[2], ego_est[3] = ego[2], ego[3]

        # 4. forecast + longitudinal control (adaptive cruise + emergency brake)
        ttc = forecast_ttc(ego_est, tracks, cfg.horizon, cfg.dt, cfg.crash_dist)
        ehead = np.array([np.cos(ego_est[2]), np.sin(ego_est[2])])
        eleft = np.array([-np.sin(ego_est[2]), np.cos(ego_est[2])])
        lead_gap, lead_speed = np.inf, cfg.target_speed
        for tr in tracks:
            rel = tr.pos() - ego_est[:2]
            fwd, lat = rel @ ehead, rel @ eleft
            if fwd > 0 and abs(lat) < 2.0 and fwd < lead_gap:
                lead_gap, lead_speed = fwd, tr.vel() @ ehead
        braking = False
        a_cmd = np.clip(cfg.target_speed - ego[3], -cfg.max_decel, cfg.max_accel)
        if cfg.avoidance:
            if np.isfinite(lead_gap):
                desired_gap = 6.0 + 1.2 * ego[3]
                a_acc = 0.6 * (lead_gap - desired_gap) + 1.0 * (lead_speed - ego[3])
                a_cmd = min(a_cmd, np.clip(a_acc, -cfg.max_decel, cfg.max_accel))
            if ttc < cfg.ttc_threshold:
                a_cmd, braking = -cfg.max_decel, True

        # steering (N4 controller acts on the believed state)
        if cfg.controller == "stanley":
            delta = stanley_steering(ego_est, path, cfg.stanley_k)
        else:
            la, _ = find_lookahead_point(ego_est, path, cfg.lookahead)
            delta = pure_pursuit_steering(ego_est, la)

        # end of path: brake to a stop and hold heading, instead of circling the last waypoint
        if np.hypot(ego[0] - path[-1, 0], ego[1] - path[-1, 1]) < cfg.lookahead:
            delta = 0.0
            a_cmd = np.clip(-ego[3] / cfg.dt, -cfg.max_decel, 0.0)

        # 5. step + crash check (true states)
        ego = bicycle_step(ego, delta, a_cmd, cfg.dt)
        min_dist = min(np.hypot(ego[0]-a[0], ego[1]-a[1]) for a in agents)
        if min_dist < cfg.crash_dist:
            crashed = True

        log["t"].append(t); log["ego"].append(ego.copy()); log["ego_est"].append(ego_est.copy())
        log["antag"].append(antag_w.copy()); log["ambient"].append(ambient_w.copy())
        log["tracks"].append([{"id": tr.id, "pos": tr.pos().copy(), "vel": tr.vel().copy(),
                               "cov": tr.P[:2, :2].copy(), "state": tr.state,
                               "color": tr.color} for tr in tracks])
        log["ttc"].append(ttc); log["min_dist"].append(min_dist)
        log["cte"].append(cross_track_error(ego, path))
        log["braking"].append(braking); log["crash"].append(crashed)

    for kk in ("ego","ego_est","antag","ambient"):
        log[kk] = np.array(log[kk])
    for kk in ("t","ttc","min_dist","cte"):
        log[kk] = np.array(log[kk])
    log["crashed"] = crashed; log["path"] = path
    return log


# quick sanity run
_demo = simulate(SimConfig(scenario="cut_in", avoidance=False))
print(f"cut-in, avoidance OFF -> crashed={_demo['crashed']}, "
      f"min clearance={_demo['min_dist'].min():.2f} m")

## 9. Looking at one run

Before animating, let's plot a single scenario statically: the BEV trajectories
on the left, and the diagnostics (speed, cross-track error, TTC, min distance)
on the right. Compare avoidance **off** vs **on** below.

In [ ]:
def plot_run(log, title=""):
    fig, (axm, axd) = plt.subplots(1, 2, figsize=(17, 7),
                                   gridspec_kw={"width_ratios": [3, 2]})
    p = log["path"]
    axm.plot(p[:, 0], p[:, 1], color="lightgray", lw=8, alpha=0.6, zorder=0)
    axm.plot(log["ego"][:, 0], log["ego"][:, 1], "b-", lw=2, label="ego")
    axm.plot(log["antag"][:, 0], log["antag"][:, 1], "r-", lw=2, label="antagonist")
    axm.plot(log["ambient"][:, 0], log["ambient"][:, 1], "-", color="orange", lw=2, label="ambient")
    if log["crashed"]:
        i = int(np.argmin(log["min_dist"]))
        axm.plot(log["ego"][i, 0], log["ego"][i, 1], "x", color="red", ms=22, mew=4, zorder=9)
        axm.text(log["ego"][i, 0], log["ego"][i, 1] + 3, "CRASH", color="red",
                 fontsize=13, fontweight="bold", ha="center")
    axm.set_aspect("equal"); axm.grid(alpha=0.3); axm.legend(loc="upper left")
    axm.set_xlabel("X — forward (m)"); axm.set_ylabel("Y — left (m)")
    axm.set_title((title + "  ") + ("CRASH" if log["crashed"] else "no crash"))

    t = log["t"]
    axd.plot(t, log["ego"][:, 3], "b-", label="ego speed (m/s)")
    axd.plot(t, log["min_dist"], "k-", label="min distance (m)")
    ttc = np.where(np.isfinite(log["ttc"]), log["ttc"], np.nan)
    axd.plot(t, ttc, "m-", label="TTC (s)")
    axd.plot(t, np.abs(log["cte"]), "g-", alpha=0.6, label="|cross-track err| (m)")
    axd.axhline(0, color="gray", lw=0.5)
    br = np.array(log["braking"], bool)
    axd.fill_between(t, 0, 1, where=br, transform=axd.get_xaxis_transform(),
                     color="red", alpha=0.12, label="braking")
    axd.set_xlabel("time (s)"); axd.set_ylim(0, 12); axd.grid(alpha=0.3); axd.legend(loc="upper right")
    axd.set_title("Diagnostics")
    plt.tight_layout(); plt.show()

plot_run(simulate(SimConfig(scenario="cut_in", avoidance=False)), "Cut-in, avoidance OFF:")
plot_run(simulate(SimConfig(scenario="cut_in", avoidance=True)),  "Cut-in, avoidance ON:")

## 10. The BEV animation

Now the same run, animated from above. The ego is blue, the antagonist red, the
ambient car orange. Faint ellipses are the **tracker's uncertainty** about each
car (N2/N3); the dashed line is the ego's **lookahead point** (N4). The banner
shows TTC and flashes when the safety layer brakes — and the scene flashes red on
a crash.

In [ ]:
def vehicle_box(ax, x, y, psi, color, length=4.5, width=2.0, **kw):
    rect = mpatches.Rectangle((-length/2, -width/2), length, width,
                              facecolor=color, edgecolor="black", lw=1, **kw)
    rect.set_transform(Affine2D().rotate(psi).translate(x, y) + ax.transData)
    ax.add_patch(rect)
    return rect


def cov_ellipse(ax, mean, cov, color):
    vals, vecs = np.linalg.eigh(cov)
    vals = np.clip(vals, 1e-6, None)
    ang = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    w, h = 2 * 2 * np.sqrt(vals)        # 2-sigma
    e = mpatches.Ellipse(mean, w, h, angle=ang, facecolor="none",
                         edgecolor=color, lw=1.2, alpha=0.7)
    ax.add_patch(e); return e


def animate_run(log, step=2):
    p = log["path"]
    idxs = list(range(0, len(log["t"]), step))
    fig, ax = plt.subplots(figsize=(12, 7))

    def draw(fi):
        ax.clear()
        k = idxs[fi]
        ax.plot(p[:, 0], p[:, 1], color="lightgray", lw=10, alpha=0.5, zorder=0)
        # trails
        ax.plot(log["ego"][:k+1, 0], log["ego"][:k+1, 1], "b-", lw=1.5, alpha=0.7)
        # tracker uncertainty + ids
        for tr in log["tracks"][k]:
            cov_ellipse(ax, tr["pos"], tr["cov"], tr["color"])
            ax.text(tr["pos"][0], tr["pos"][1] + 1.6, f"ID{tr['id']}",
                    color=tr["color"], fontsize=8, ha="center")
        # vehicles
        e = log["ego"][k]; vehicle_box(ax, e[0], e[1], e[2], "tab:blue", zorder=5)
        a = log["antag"][k]; vehicle_box(ax, a[0], a[1], a[2], "tab:red", zorder=5)
        b = log["ambient"][k]; vehicle_box(ax, b[0], b[1], b[2], "tab:orange", zorder=5)
        # lookahead
        la, _ = find_lookahead_point(log["ego_est"][k], p, 8.0)
        ax.plot([e[0], la[0]], [e[1], la[1]], "b--", lw=1, alpha=0.5)
        ax.plot(la[0], la[1], "bo", ms=5)
        # banner
        ttc = log["ttc"][k]
        msg = f"t={log['t'][k]:.1f}s  v={e[3]:.1f} m/s  TTC={'inf' if not np.isfinite(ttc) else f'{ttc:.1f}s'}"
        if log["braking"][k]: msg += "  BRAKING"
        ax.text(0.02, 0.97, msg, transform=ax.transAxes, va="top", fontsize=11,
                fontweight="bold", bbox=dict(boxstyle="round", fc="white", alpha=0.8))
        if log["crash"][k]:
            ax.text(0.5, 0.5, "CRASH", transform=ax.transAxes, ha="center", va="center",
                    fontsize=40, color="red", fontweight="bold", alpha=0.6)
        # view window follows the ego
        ax.set_xlim(e[0] - 35, e[0] + 35); ax.set_ylim(e[1] - 25, e[1] + 25)
        ax.set_aspect("equal"); ax.grid(alpha=0.3)
        ax.set_xlabel("X — forward (m)"); ax.set_ylabel("Y — left (m)")

    anim = FuncAnimation(fig, draw, frames=len(idxs), interval=80)
    plt.close(fig)
    return HTML(anim.to_jshtml())

animate_run(simulate(SimConfig(scenario="cut_in", avoidance=True)))

## 11. Your turn — tune the knobs and render

Set the controls, then click **▶ Render run** (or a preset). Each render simulates the
scenario from scratch and plays it back as a **smooth bird's-eye animation**, with a
diagnostics panel underneath. Rendering takes a couple of seconds; the playback itself is
smooth. Re-render whenever you change a setting.

Things to try:
- *Cut-in → crash*: turn **avoidance** off and watch the red car merge into you.
- *Cut-in → avoided*: turn it back on — the ego brakes in time.
- Load **Cut-in → avoided**, then lower the **TTC thresh** toward 0.5 s — the ego
  reacts too late and you can watch it cross back into a crash.
- *Sensor blackout*: avoidance on, but open a **dropout** window over the cut-in —
  the ego is blind exactly when it matters, and crashes anyway.
- Drop **lookahead** to ~2 m and watch the ego oscillate (N4).

In [ ]:
import ipywidgets as W
from IPython.display import display

# --- driving parameters ---
ctrl   = W.Dropdown(options=["pure_pursuit", "stanley"], value="pure_pursuit", description="controller")
look   = W.FloatSlider(value=8, min=2, max=20, step=1, description="lookahead")
speed  = W.FloatSlider(value=8, min=3, max=14, step=1, description="ego speed")
avoid  = W.Checkbox(value=True, description="avoidance")
ttc_th = W.FloatSlider(value=3, min=0.5, max=6, step=0.5, description="TTC thresh")
scen   = W.Dropdown(options=["cut_in", "sudden_stop", "none"], value="cut_in", description="scenario")
trig   = W.FloatSlider(value=5, min=1, max=12, step=0.5, description="trigger t")
asp    = W.FloatSlider(value=6, min=0, max=12, step=1, description="antag speed")
noise  = W.FloatSlider(value=0.4, min=0, max=3, step=0.1, description="sensor noise")
drop   = W.FloatRangeSlider(value=[0, 0], min=0, max=22, step=0.5, description="dropout")

def _cfg():
    d0, d1 = drop.value
    return SimConfig(controller=ctrl.value, lookahead=look.value, target_speed=speed.value,
                     avoidance=avoid.value, ttc_threshold=ttc_th.value, scenario=scen.value,
                     antagonist_trigger=trig.value, antagonist_speed=asp.value, sensor_noise=noise.value,
                     dropout_start=(d0 if d1 > d0 else -1.0), dropout_end=(d1 if d1 > d0 else -1.0))

# render-on-demand: simulate once per click and play back a pre-rendered (smooth) animation,
# instead of redrawing every frame live in Python (which was laggy).
out = W.Output()

def render(_=None):
    with out:
        out.clear_output(wait=True)
        cfg = _cfg()
        log = simulate(cfg)
        tag = "CRASH" if log["crashed"] else "no crash"
        print(f"{tag}   min clearance = {log['min_dist'].min():.2f} m"
              + ("   (safety layer braked)" if any(log["braking"]) else ""))
        display(animate_run(log))   # smooth, pre-rendered BEV video
        plot_run(log)               # static diagnostics

run_btn = W.Button(description="▶ Render run", button_style="success")
run_btn.on_click(render)

def preset(name):
    def _apply(_):
        scen.value = "none" if name == "clean" else "cut_in"
        avoid.value = (name != "crash")
        trig.value = 5; asp.value = 6
        drop.value = [4.5, 9.0] if name == "blackout" else [0, 0]
        render()
    return _apply

p_clean = W.Button(description="Clean drive");       p_clean.on_click(preset("clean"))
p_crash = W.Button(description="Cut-in -> crash");    p_crash.on_click(preset("crash"))
p_avoid = W.Button(description="Cut-in -> avoided");  p_avoid.on_click(preset("avoid"))
p_black = W.Button(description="Sensor blackout");    p_black.on_click(preset("blackout"))

controls = W.VBox([
    W.HBox([ctrl, look, speed]),
    W.HBox([avoid, ttc_th]),
    W.HBox([scen, trig, asp]),
    W.HBox([noise, drop]),
    W.HBox([run_btn, W.Label("set the knobs, then render")]),
    W.HBox([W.Label("Presets:"), p_clean, p_crash, p_avoid, p_black]),
])
display(controls, out)
render()   # initial render so the cell isn't blank


## 12. (Optional bonus) Seeding the scene from *real* detections (an N3-style track feed)

Everything above seeds agents from scripted lanes. As a bonus that closes the
loop back to N1/N3, we can instead drop a real car into the BEV frame: run YOLO
on a KITTI camera frame, take a vehicle's box, and **back-project** its
ground-contact point to the BEV ground plane (N1's `backproject_to_ground`).

This cell is **optional and guarded** — it needs the KITTI images plus the
~237 MB YOLOv3 weights (the same detector as Workshop 4.2). If they aren't present it just explains what it
would do and skips. The scripted scenarios above don't depend on it.

In [ ]:
import urllib.request
os.makedirs("models", exist_ok=True)

# Auto-fetch YOLOv3 (the same detector as Workshop 4.2) if not already in models/
_yolo_files = {
    "models/yolov3.cfg":     "https://raw.githubusercontent.com/pjreddie/darknet/master/cfg/yolov3.cfg",
    "models/yolov3.weights": "https://pjreddie.com/media/files/yolov3.weights",   # ~237 MB, one-time
}
import shutil
for _path, _url in _yolo_files.items():
    if not os.path.exists(_path):
        try:
            print(f"Downloading {_path}" + (" (~237 MB, one-time)..." if _path.endswith(".weights") else "..."))
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})  # pjreddie 403s the default UA
            with urllib.request.urlopen(_req, timeout=120) as _r, open(_path, "wb") as _f:
                shutil.copyfileobj(_r, _f)
        except Exception as _e:
            print(f"  could not download {_path}: {_e}")
            if os.path.exists(_path):
                os.remove(_path)   # drop any partial/empty file so the presence check stays honest

KITTI_PRESENT = os.path.isdir(os.path.join("kitti_data", "2011_09_26",
                               "2011_09_26_drive_0005_sync", "image_02", "data"))
WEIGHTS_PRESENT = os.path.exists("models/yolov3.weights") and os.path.exists("models/yolov3.cfg")

if not (KITTI_PRESENT and WEIGHTS_PRESENT):
    print("Bonus skipped:")
    if not KITTI_PRESENT:
        print("  - KITTI drive not found — run N1 first (it downloads the drive into kitti_data/).")
    if not WEIGHTS_PRESENT:
        print("  - YOLOv3 weights/cfg missing from models/ — the auto-download above may have failed; add them manually and re-run.")
else:
    import cv2, pykitti
    print("KITTI + YOLO present - back-projecting a real detection into the BEV scene.")
    # 1) load a frame + calibration (same as N1)
    data = pykitti.raw("kitti_data", "2011_09_26", "0005")
    img = np.array(data.get_cam2(0))
    P2 = np.array(data.calib.P_rect_20)
    R0 = np.array(data.calib.R_rect_00)
    Tr = np.array(data.calib.T_cam0_velo)
    R0e = np.eye(4); R0e[:3, :3] = R0[:3, :3] if R0.shape == (3, 3) else R0[:3, :3]
    M = P2 @ (R0 if R0.shape == (4, 4) else R0e) @ Tr

    def backproject_to_ground(uv, M, z_ground=-1.6):
        u, v = float(uv[0]), float(uv[1])
        a = M[0, 2]*z_ground + M[0, 3]; b = M[1, 2]*z_ground + M[1, 3]; c = M[2, 2]*z_ground + M[2, 3]
        A = np.array([[M[0, 0]-u*M[2, 0], M[0, 1]-u*M[2, 1]],
                      [M[1, 0]-v*M[2, 0], M[1, 1]-v*M[2, 1]]])
        rhs = np.array([u*c-a, v*c-b])
        XY = np.linalg.solve(A, rhs)
        return XY   # [X_forward, Y_left] in velodyne/BEV frame

    # 2) detect vehicles with YOLO (same detector as Workshop 4.2)
    net = cv2.dnn.readNet("models/yolov3.weights", "models/yolov3.cfg")
    ln = net.getLayerNames(); out_layers = [ln[i-1] for i in net.getUnconnectedOutLayers()]
    H, Wd = img.shape[:2]
    blob = cv2.dnn.blobFromImage(img, 1/255.0, (416, 416), swapRB=True, crop=False)
    net.setInput(blob)
    boxes = []
    for o in net.forward(out_layers):
        for det in o:
            if det[5:].max() >= 0.5 and np.argmax(det[5:]) in (2, 5, 7):  # car/bus/truck
                cx, cy, w, h = det[0]*Wd, det[1]*H, det[2]*Wd, det[3]*H
                boxes.append((cx, cy + h/2))   # bottom-center = ground contact
    bev = np.array([backproject_to_ground(b, M) for b in boxes]) if boxes else np.empty((0, 2))
    bev = bev[(bev[:, 0] > 0) & (bev[:, 0] < 60) & (np.abs(bev[:, 1]) < 15)] if len(bev) else bev

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 5))
    a1.imshow(img); a1.set_title("KITTI camera frame"); a1.axis("off")
    a2.scatter(bev[:, 0], bev[:, 1], c="red", s=80, marker="s") if len(bev) else None
    a2.plot(0, 0, "b^", ms=14); a2.set_aspect("equal"); a2.grid(alpha=0.3)
    a2.set_xlabel("X — forward (m)"); a2.set_ylabel("Y — left (m)")
    a2.set_title(f"Real detections back-projected to BEV ({len(bev)} cars)")
    plt.tight_layout(); plt.show()
    print("These real BEV positions could seed `ambient` agents in simulate().")

## Summary

This capstone closed the loop on Workshop 4.3 by fusing every prior notebook into
one interactive BEV simulator:

1. **Shared BEV frame (N1)** — a single metric ground plane, anchored on the real
   KITTI drive-0005 ego trajectory.
2. **Control (N4)** — a bicycle model + pure-pursuit/Stanley controller steers the
   ego along that path.
3. **Other-car motion** — constant-velocity, lane-relative motion drives the other cars and
   forecasts where they're going.
4. **Perception & tracking (N2 + N3)** — a per-object constant-velocity Kalman
   tracker with a full lifecycle turns noisy detections into stable, identified
   tracks, and coasts through sensor dropout.
5. **Decision-making (new)** — a TTC forecast + adaptive-cruise/emergency-brake
   safety layer decides when to act, producing the crash-vs-avoid outcomes.

### Key takeaways
- **Estimation quality gates safety.** The "sensor blackout" preset crashes even
  with avoidance on — you can't avoid what you can't perceive. This is why N2's
  estimation and N3's tracking matter, not just the controller.
- **A controller alone isn't autonomy.** N4 keeps the ego on its path beautifully,
  but without the perception → tracking → forecasting chain it drives straight
  into a stopped car.
- **Modularity pays off.** Each notebook's component slotted in behind a clean
  interface — the same lesson that lets real autonomy stacks swap detectors,
  trackers, or planners independently.

### Extensions to explore
- Replace constant-velocity track prediction with a full **EKF/CTRV** turning model for
  turning agents.
- Swap the synthetic reference path for **A\*** output from Workshop 4.2.
- Add a **lateral** avoidance maneuver (swerve) instead of braking only.
- Seed *all* agents from the real-detection bonus pipeline for a fully data-driven
  scene.
